In [ ]:
input_data = None
targetpop_data = None
output_data = None
output_model = None
util = None
display_util = None
configfile = "config/config.yml"

In [ ]:
import yaml

with open(configfile) as stream:
    config = yaml.safe_load(stream)

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib_inline
import matplotlib.pyplot as plt

from IPython.display import Markdown
import pandera.pandas as pa
from pandera.typing import Series

matplotlib_inline.backend_inline.set_matplotlib_formats("svg")

%matplotlib inline
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", None)
plt.ioff()
plt.rcParams["figure.figsize"] = (8, 5)

sys.path.append(str(Path(util).parent))
sys.path.append(str(Path(display_util).parent))

In [ ]:
from display_util import (  # noqa: E402
    rule_setup,
    display_data_doc,
    display_long_data_doc,
    collist,
)
from util import (  # noqa: E402
    drop_col_few_distinct,
    TransplantPostOPID,
    drop_duplicate_columns,
    common_translate,
    fix_units,
)

### Target Population Filtering

The transplantations in the dataset were filtered to match the target population (see [](general:tpf)). Afterwards we tried again to remove empty and duplicate columns.

In [ ]:
data = pd.read_parquet(input_data)
trans = data["transplant_et_id"]
targetpop = pd.read_parquet(targetpop_data)
data = data[trans.isin(targetpop["transplant_et_id"])]
display(
    Markdown(
        f"""The filter process reduces the number of transplantations in the data ({trans.nunique()}) and target population ({targetpop["transplant_et_id"].nunique()})
            to {trans[trans.isin(targetpop["transplant_et_id"])].nunique()} in the processed data.
        """
    )
)

In [ ]:
data = drop_col_few_distinct(data)
data = drop_duplicate_columns(data)

### Integration of Seperated Institute Data

Only the {term} `ET` contributed to this dataset (see [](general:ic)). 

## Domain Steps

For this file the general plan for domain preprocessing of longitudinal data was followed (see [](general:ds)).

### Row Filtering

There is no column differentiating between different types of short term follow-up (see [](general:rf)). We kept all rows.

In [ ]:
display_long_data_doc(data, ["transplant_et_id"], "examination_date", None)

### Unit Conversions

Common translations were applied and then the creatinine measurements converted to a single unit (see [](general:uc)). Single unit specifiers were removed afterwards.

In [ ]:
data = common_translate(data, config["data"]["common_translations"])
# Maybe add to config more if necessary

In [ ]:
fix_units(
    data,
    "creatinine_mg_per_dl",
    "creatinine_unit",
    config["data"]["unit_conversions"]["creatinine"]["target"],
    config["data"]["unit_conversions"]["creatinine"]["factors"],
)

In [ ]:
cols = data.columns[data.columns.to_series().str.contains("_unit")]
assert (data[cols].nunique() != 1).sum() == 0
dropme = cols[data[cols].nunique() == 1]
data = data.drop(columns=dropme)
display(
    Markdown(
        f"The columns {collist(dropme)} were removed as only a single unit was used."
    )
)

### Consolidating Columns

No consolidation was necessary. (see [](general:crc))

## Intermediate Dataset

For this longitudinal dataset we recommend the `examination_date` column as the time axis.

In [ ]:
indcols = ["transplant_et_id"]
data = data.sort_index(axis=1).sort_values(indcols + ["examination_date"], axis=0)
data = data.set_index(indcols)

In [ ]:
# Another base class might be necessary, see util.py
# describe columns, without checks for now, order is important
class TransplantationPostOPExamination(TransplantPostOPID):
    creatinine_mg_per_dl: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Creatinine",
        description="What was the patients Creatinine measurement in mg/dl.",
    )
    examination_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Examination date",
        description="Date when the lab measurements were conducted",
    )

    class Config:
        title = "Transplantation post operation examination dataset"
        description = "Each row represents a follow-up. The data is based on the 'element_transplantation_postop_untersuchung.csv' file. It contains data from the ET."
        multiindex_strict = True
        multiindex_coerce = True

In [ ]:
display_data_doc(TransplantationPostOPExamination, data)

In [ ]:
TransplantationPostOPExamination.to_schema().validate(data).to_parquet(output_data)
with open(output_model, "wt") as fh:
    TransplantationPostOPExamination.to_yaml(stream=fh)

## Technical Information

In [ ]:
rule_setup(
    {
        "input_data": input_data,
        "targetpop_data": targetpop_data,
        "output_data": output_data,
        "output_model": output_model,
        "util": util,
        "display_util": display_util,
    }
)